## **Lo que van a encontrar en este notebook**

Tenemos dos metas:

1. Ver ejemplos de curvas caracteristicas de Euler, utilizandolas para inferir la topologia de los datos y entrenar un clasificador de objetos geometricos reales (TOSCA dataset)

2. Aplicar clustering topologico a las curvas caracteristicas de Euler de los modelos TOSCA 

Completar las actividades marcadas con **Hacer** o **Tu Respuesta**. Sugiero leer atentamente el codigo y los comentarios para tener una idea de que esta pasando.

Autores: Matt Piekenbrock, Jose Perea

In [ ]:
!pip install pymeshlab

In [ ]:
import numpy as np
np.seterr(invalid='ignore')
import scipy.io

def tosca():
    import tarfile
    from urllib.request import urlopen
    from io import BytesIO
    url = "https://raw.githubusercontent.com/peekxc/tosca_signatures/main/tosca.tar.xz"
    tosca_file = tarfile.open(fileobj=BytesIO(urlopen(url).read()) , mode="r:xz")
    tosca_files = [name[:-4] for name in tosca_file.getnames() if name[0] != "."]
    def _extract_model(model_name: str, obj_id: str = None, normalize: bool = True):
        fn = (model_name + str(obj_id) + ".mat") if obj_id is not None else (model_name + ".mat")
        m = tosca_file.extractfile(fn)
        mat = scipy.io.loadmat(BytesIO(m.read()))
        x = np.ravel(mat['surface']['X'][0][0]).astype(float)
        y = np.ravel(mat['surface']['Y'][0][0]).astype(float)
        z = np.ravel(mat['surface']['Z'][0][0]).astype(float)
        S = np.c_[x,y,z]
        T = mat['surface']['TRIV'][0][0] - 1 # TOSCA is 1-based
        if normalize:
            S -= S.mean(axis=0)
            c = np.linalg.norm(S.min(axis=0) - S.max(axis=0))
            S *= (1/c)
        return S, T
    return _extract_model, tosca_files

La funcion `tosca()` retorna:

1.  Una funcion para cargar modeleos (mallas - complejos simpliciales 2-dimensionales)
2.  El conjunto de modelos disponibles en el conjunto de datos



In [ ]:
get_tosca_model, model_names = tosca()

Por ejemplo, para cargar la malla correspondiente a `wolf1` podemos usar:

In [ ]:
X, T = get_tosca_model("wolf1")

Esto retorna una dupla (`X`, `T`) donde `X`=vertex_coordinates (conjunto de vertices) `T`=triangle_indices (triangulos)

In [ ]:
print(f"  shape: X={X.shape}, T={T.shape}")

En el modelo `wolf1` hay 4.344 vertices y 8.684 triangulos.

A continuacion tenemos codigo para seleccionar algunos modelos aleatoriamente y visualizarlos

In [ ]:
import matplotlib.pyplot as plt
from itertools import product
np.random.seed(234)
nrow, ncol = 1, 4
zoom = 0.25
tosca_models = np.random.choice(model_names, size=nrow*ncol, replace=False)

fig, axs = plt.subplots(nrow, ncol, figsize=(80, 30), subplot_kw={'projection': '3d'})
idx_iter = range(nrow*ncol)
for model_name, i in zip(tosca_models, idx_iter):
    X, T = get_tosca_model(model_name)
    c, rng = X.mean(axis=0), max(abs(X.max(axis=0)-X.min(axis=0)))
    axs[i].scatter(*X.T, s=18, c=X[:,0], cmap='plasma')
    axs[i].set_xlim(c[0] - zoom*rng, c[0] + zoom*rng)
    axs[i].set_ylim(c[1] - zoom*rng, c[1] + zoom*rng)
    axs[i].set_zlim(c[2] - zoom*rng, c[2] + zoom*rng)
    axs[i].axis('off')
plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

---

## Curvas caracteristicas de Euler

La caracteristica de Euler de un complejo simplicial 2-dimensional  $K$ es:
$$ \chi(K) = \#(V) - \#(E) + \#(T) $$
donde ($V$, $E$, $T$) denotan los vertices, aristas y  triangulos de $K$, respectivamente.

Sea $f: V \rightarrow \mathbb{R}$  una funcion y para $\alpha \in \mathbb{R} $ sea 

$$K_\alpha = \left\{\sigma \in K \; : \; \max \big(f(\sigma)\big) \leq \alpha\right\} $$

Si  $n = \#(K)$  y $  \alpha_1 < \alpha_2  < \cdots < \alpha_m $ entonces la _curva caracteristica de Euler_ $ \alpha_i \mapsto \chi(K_{\alpha_i})$  se puede calcular de forma no sofisticada (`simple`)  en  tiempo $O(nm)$. Afortundamente, existe un algoritmo mas efectivo (`top_down`) que la calcula en $O(n + m)$.

In [ ]:
from numpy.typing import ArrayLike
from typing import Collection, Union

def euler_curve(X: ArrayLike, T: ArrayLike, f: ArrayLike, bins: Union[int, Collection[float]] = 20, method: str = "simple") -> ArrayLike:
    """Calculates the euler characteristic curve.
    Parameters:
        X = (n x 3) array of vertex coordinates
        T = (m x 3) array of triangle indices
        f = array of vertex values to filter by
        bins = number of bins, or a sequence of threshold values
        method = "simple" or "top_down" (both numpy-only, no external deps)
    Returns:
        array of euler characteristic values of the mesh along _bins_
    """
    assert isinstance(X, np.ndarray) and X.ndim == 2, "Invalid point cloud given."
    assert isinstance(T, np.ndarray) and T.ndim == 2 and T.shape[1] == 3, "Invalid triangles given."
    f = np.array(f)
    assert isinstance(f, np.ndarray) and len(f) == X.shape[0], "Invalid vertex function array given."
    bins = np.linspace(min(f), max(f) + 10 * np.finfo(float).eps, bins) if isinstance(bins, int) else bins
    assert isinstance(bins, Collection), f"Invalid argument bins={bins}"

    ft = f[T].max(axis=1)  # triangle weights (max of the triangle's own 3 vertex values)

    if method == "simple":
        edges = np.vstack([T[:, [0, 1]], T[:, [0, 2]], T[:, [1, 2]]])
        unique_edges = np.unique(np.sort(edges, axis=1), axis=0)  # canonicalized, properly deduped
        fe = f[unique_edges].max(axis=1)
        ecc = np.array([sum(f < t) - sum(fe < t) + sum(ft < t) for t in bins])
    elif method == "top_down":
        # hirola-free: replaces HashTable with np.unique for edge dedup (no compiled-extension crash risk)
        vw = f  # vertex weight is just the vertex's own value
        edges = np.vstack([T[:, [0, 1]], T[:, [0, 2]], T[:, [1, 2]]])
        unique_edges = np.unique(np.sort(edges, axis=1), axis=0)
        ew = f[unique_edges].max(axis=1)
        v_counts = np.cumsum(np.histogram(vw, bins=bins)[0])
        e_counts = np.cumsum(np.histogram(ew, bins=bins)[0])
        t_counts = np.cumsum(np.histogram(ft, bins=bins)[0])
        ecc = np.append(0, (v_counts - e_counts + t_counts))
    else:
        raise ValueError(f"Unknown method {method}")
    return ecc

`top_down` es una version optimizada del calculo `simple`, como lo muestra la siguiente comparacion:

In [ ]:
import timeit
X, T = get_tosca_model("wolf1")
print('Tiempo - Curva caracteristica de Euler (simple):')
print(timeit.timeit(lambda: euler_curve(X, T, X @ np.array([1,0,0]), method="simple"), number=5))
print('Tiempo - Curva caracteristica de Euler (top_down):')
print(timeit.timeit(lambda: euler_curve(X, T, X @ np.array([1,0,0]), method="top_down"), number=5))

---

Veamos ahora algunos de las clases de mallas en el TOSCA dataset

In [ ]:
import re
cls_rgx = re.compile(r"^([a-zA-Z]+)(\d+)") ## class regex
tosca_classes = ['dog', 'cat', 'michael', 'centaur', 'victoria', 'horse', 'david', 'gorilla', 'wolf']
tosca_colors = ['red', 'blue', 'black', 'orange', 'yellow', 'green', 'purple', 'pink', 'cyan']
tosca_class_ids = { cls_nm : [] for cls_nm in  tosca_classes}
for fn in model_names:
    cls_nm, model_id = cls_rgx.split(fn)[1:3]
    tosca_class_ids[cls_nm].append(int(model_id))

import pprint
pprint.PrettyPrinter(indent=2, compact=True).pprint(tosca_class_ids)

y las **curvas caracteristicas de Euler** usando la direccion horizontal (eje $x$) en las mallas

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()
# tosca_subset = np.random.choice(tosca_classes, size = 4, replace=False)
# tosca_subset = ["dog", "cat", "horse", "centaur", "victoria"]
tosca_subset = ["michael", "david", "victoria", "dog"]
bins = 20
INDEX = np.arange(bins)

p = figure(width=750, height=300, x_axis_label="Valor de alpha - coordenada x", y_axis_label="Caracteristica de Euler")
for tc in tosca_subset:
    for model_id in tosca_class_ids[tc]:
        X, T = get_tosca_model(tc, model_id)
        f = X @ np.array([1,0,0])
        ecc = euler_curve(X, T, f, method="top_down")
        p.line(INDEX, ecc, color=tosca_colors[tosca_classes.index(tc)], legend_label=tc)
p.legend.location = "bottom_left"
show(p)

## Usando informacion geometrica / curvatura

Las curvaturas principales $\kappa_1$ y $\kappa_2$ son la base de muchos descriptores geometricos. Por ejemplo la curvatura de Gauss $\kappa$ y la curvatura media $H$, se definen como:

$$ \kappa = \kappa_1  \kappa_2, \quad H = (\kappa_1 + \kappa_2)/2 $$

Otro descriptor utilizado en la pratica (dada su invariancia a escala) es el  _indice de forma_ $s$:

$$ s = \frac{2}{\pi} \mathrm{arctan} \frac{\kappa_2 + \kappa_1}{\kappa_2 - \kappa_1}, \quad \kappa_1 \geq \kappa_2 $$

In [ ]:
import pymeshlab
def curvature(
    X: ArrayLike,
    T: ArrayLike,
    value=["mean", "gaussian", "min", "max", "shape_index", "curvedness"],
    fit=['Quadric Fitting', 'Normal Cycles', 'Taubin approximation']
) -> ArrayLike:
    """uses pymeshlab to compute the principal directions of curvature with different algorithms.

    Parameters:
        X = (n x 3) array of vertex coordinates
        T = (m x 3) array of triangle indices
        value = type of curvature to compute.
        fit = algorithm to fit the curvature by.

    Returns:
        array of curvature values at the mesh vertices.

    Note: This function can fail if the mesh has non-unique vertices

    See: https://pymeshlab.readthedocs.io/en/2021.10/filter_list.html#compute_curvature_principal_directions for more details.
    """
    fit_method = {'quadric fitting' : 3, 'normal cycles' : 2, 'taubin approximation' : 0 }
    value_type = {'mean': 0, 'gauss': 1, 'min': 2, 'max': 3, 'shape_index': 4, 'curvedness': 5 }
    value = 'shape_index' if isinstance(value, list) else str(value).tolower()
    fit = 'quadric fitting' if isinstance(fit, list) else str(fit).tolower()
    assert isinstance(fit, str) and fit in fit_method.keys(), f"Invalid fit method '{fit}' given."
    assert isinstance(value, str) and value in value_type.keys(), f"Invalid curvature type '{value}' given."
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(X, T))
    ms.compute_curvature_principal_directions_per_vertex(method=3, curvcolormethod=4)
    mesh = ms.mesh(0)
    return mesh.vertex_scalar_array()

A continuacion visualizamos algunas de las mallas coloreadas de acuerdo al indice de forma ($s$)

In [ ]:
np.random.seed(234)
nrow, ncol = 1, 4
zoom = 0.25
tosca_models = np.random.choice(model_names, size=nrow*ncol, replace=False)

fig, axs = plt.subplots(nrow, ncol, figsize=(80, 30), subplot_kw={'projection': '3d'})
idx_iter = range(nrow*ncol)
for model_name, i in zip(tosca_models, idx_iter):
    X, T = get_tosca_model(model_name)
    c, rng = X.mean(axis=0), max(abs(X.max(axis=0)-X.min(axis=0)))
    f = curvature(X, T)
    axs[i].scatter(*X.T, s=80, c=f, cmap='plasma')
    axs[i].set_xlim(c[0] - zoom*rng, c[0] + zoom*rng)
    axs[i].set_ylim(c[1] - zoom*rng, c[1] + zoom*rng)
    axs[i].set_zlim(c[2] - zoom*rng, c[2] + zoom*rng)
    axs[i].axis('off')
plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)


Las **curvas caracteristicas de Euler** con respecto al indice de forma $s$ son

In [ ]:
tosca_subset = ["michael", "david", "victoria", "dog"]
p = figure(width=750, height=300, x_axis_label="Valor de alpha - indice de forma", y_axis_label="Caracteristica de Euler")
for tc in tosca_subset:
    for model_id in tosca_class_ids[tc]:
        X, T = get_tosca_model(tc, model_id)
        s = curvature(X, T)
        ecc = euler_curve(X, T, s, method="top_down")
        p.line(INDEX, ecc, color=tosca_colors[tosca_classes.index(tc)], legend_label=tc)
p.legend.location = "bottom_left"
show(p)

dadas las curvas arriba, cualquier modelo de clasificacion (e.g., agrupamiento/clustering) puede clasificar las figuras geometricas.

**Hacer:** En la celda de abajo, aplica clustering topologico a las curvas caracteristicas de Euler con respecto al indice de forma $s$ para los modelos TOSCA. Tu clustering coincide con las clases? 

In [ ]:
# tu codigo aqui







---

Aca van algunos articulos que implementan y usan las curvas de Euler para aplicaciones; [2] lo hace para biologia de plantas y [3] para problemas en quimica:

## Bibliografia

1. [Efficient classification using the Euler characteristic](https://www.cs.huji.ac.il/w~werman/Papers/1-s2.0-S0167865514002050-main.pdf), Richardson and Werman, **Pattern Recognition Letters**, 49, 99 - 106, 2014.

1. [Measuring hidden phenotype: quantifying the shape of barley seeds using the Euler characteristic transform](https://watermark02.silverchair.com/diab033.pdf?token=AQECAHi208BE49Ooan9kkhW_Ercy7Dm3ZL_9Cf3qfKAc485ysgAAA4AwggN8BgkqhkiG9w0BBwagggNtMIIDaQIBADCCA2IGCSqGSIb3DQEHATAeBglghkgBZQMEAS4wEQQMu56MwiEoMtn5u7MPAgEQgIIDM5KChqfNbvZJgAw5N3Cuu3qX_DhJbr6qVB0wjgBx5oqoAzqIIGr80rYsbYBMXO9A6EBaI0dVM3WOseOPTO39wg2Do1IhZrJkFMInywp6IbRq4x2EcnaoDmG-h3LUIjY8EhuW07uV09XpFJnBBqBOyXpqQRSqfaGO2r4Gbhxqgq_8bObe1A-DzesolyyDRAH0PjmGTJMxrwmU4dHclr2HLr0jvFOoudJELJ2w7CT8yHOw5N0Fy7RQAoDTK-IYwANTLybgRZ2Er8N-tmTjhF5EYpKe7Ss7I08QHzayMbLLeJrfXyQ9FZU_-8MWRfpzGEfvusXDEB42sOJJQl85xuzdO-6FOpA9Guv3WmloFZNOsXzn7TJ_UsSszDAiZcjD6eN_WMsnItWmQmY0Yw0dr6pA3OuMKRMznaMDKryjBtko1NyaIAJimFg5As0-vLqlKSLQtaTwak0UVwd7JL8sLfqX6q676bMdpfw9W9W5C1nPXzCdQIkenxHdDNxNGruk-bqoJNmn0ELeubbdLwABGOn8DzPAw5g4KjhWcqmWNz6wnNJH3AKanQt-NXKnVLzp0AugtxDLrFGa-MtB6Zv8vaeLTzzzMqa6C3PzF4kz_G1sFn0TlTdxQ-SsQG_9VfchT3rIJeKzQImXJvfcqfKDIWd1lmDDZZNCVYOFhXhIFs66xI5otF96-3hLm_OJ-ukzJXDtzWJ76GHA6cYBB2N4hNoQmqDMM_L6kBrl-esYhQHBkwecXRaQMlZAY_dDmMz_QMl-teOjLpUGd268jfqIDQo6UYF7SE6odQuMwMknJdfwSdMNrYclkASsKYqvf5bhsaOyh_RWO0dwOWEPIcru9RJD6MnsX1Um2Gowoa3xUUEpPLZ4tV0cb9kjMMd-4vRtrGSDEpphYJ6U-WdpbWpyUoWTH2j7bjQRjTDDUaQ8Jujl7TaddXRkjBnv6jcQjKoshGsk2-dS6q2mjIGtnLz1i9sCwztxfPCnGVgTWQBd2KCuHYOcjv1NY1sj5OPhku5mLwL4oeaurUIdFqH3cm5QSRbnvH85ks0aE08u2ZmozZWTHbO58jClVNXBFUUrT1wlmM00xnPNhA), Amezquita et. al., **in silico Plants** Vol. 4, No. 1, pp. 1-15, 2021.

1. [A fast and scalable computational topology framework for the Euler characteristic](https://arxiv.org/pdf/2311.11740), Laky and Zavala, Digital Discovery - **Royal Society of Chemistry**, 3, 392-409, 2024.